In [2]:
"""
Morris Sampling for PFLOTRAN Sensitivity Analysis
===================================================
Generates parameter combinations for Morris method using SALib.
All parameters are sampled in log10 space since ranges span orders of magnitude.
"""

import numpy as np
from SALib.sample import morris as morris_sample
from SALib import ProblemSpec

In [3]:
# =============================================================================
# Step 1: Define the problem
# =============================================================================
# SALib requires a 'problem' dictionary with parameter names and bounds.
# We define bounds in log10 space so that Morris discretizes uniformly
# across orders of magnitude.

problem = {
    'num_vars': 16,
    'names': [
        'calcite_rate',
        'fh_rmax',          # ferrihydrite DIR max rate
        'fh_k_donor',       # ferrihydrite DIR acetate half-sat
        'fh_SA',            # ferrihydrite surface area
        'gt_rmax',          # goethite DIR max rate
        'gt_k_donor',       # goethite DIR acetate half-sat
        'gt_SA',            # goethite surface area
        'root_resp',        # root respiration rate
        'msr_rmax',         # sulfate reduction max rate
        'msr_k_donor',      # sulfate reduction acetate half-sat
        'msr_k_acceptor',   # sulfate reduction sulfate half-sat
        'denit_rmax',       # denitrification max rate
        'denit_k_donor',    # denitrification acetate half-sat
        'denit_k_acceptor', # denitrification nitrate half-sat
        'aero_rate',        # aerobic respiration rate constant
        'aero_k_o2',        # aerobic respiration O2 half-sat
    ],
    # Bounds in log10 space
    'bounds': [
        [np.log10(1.55e-07), np.log10(1.55e-05)],   # calcite rate
        [np.log10(2.50e-07), np.log10(2.50e-06)],   # fh_rmax**
        [np.log10(1.00e-06), np.log10(1.00e-05)],   # fh_k_donor**
        [np.log10(1.08e+06), np.log10(1.08e+08)],   # fh_SA
        [np.log10(2.50e-08), np.log10(2.50e-06)],   # gt_rmax
        [np.log10(1.00e-07), np.log10(1.00e-05)],   # gt_k_donor
        [np.log10(2.83e+05), np.log10(2.83e+07)],   # gt_SA
        [np.log10(7.94e-14), np.log10(7.94e-12)],   # root_resp
        [np.log10(2.50e-08), np.log10(2.50e-06)],   # msr_rmax
        [np.log10(5.00e-07), np.log10(5.00e-05)],   # msr_k_donor
        [np.log10(1.00e-05), np.log10(1.00e-03)],   # msr_k_acceptor
        [np.log10(1.00e-07), np.log10(1.00e-05)],   # denit_rmax
        [np.log10(1.00e-06), np.log10(1.00e-04)],   # denit_k_donor
        [np.log10(1.00e-06), np.log10(1.00e-04)],   # denit_k_acceptor
        [np.log10(1.00e-10), np.log10(1.00e-08)],   # aero_rate
        [np.log10(1.00e-05), np.log10(1.00e-03)],   # aero_k_o2
    ]
}

In [4]:
# =============================================================================
# Step 2: Generate Morris trajectories
# =============================================================================
# N = number of trajectories (r in Morris notation)
# num_levels = number of discrete levels per parameter (p)
# optimal_trajectories = number of candidate trajectories to generate and
#     select from to maximize spread (set to None for pure random)

param_values_log = morris_sample.sample(
    problem,
    N=25,                       # 25 trajectories
    num_levels=4,               # 4 discrete levels per parameter
    optimal_trajectories=None,  # set to an integer to optimize (e.g., 10)
    seed=42                     # for reproducibility
)

print(f"Sample matrix shape: {param_values_log.shape}")
# Expected: (340, 16) — that is, 20 * (16+1) = 340 rows

Sample matrix shape: (425, 16)


In [5]:
# =============================================================================
# Step 3: Convert from log10 space back to real parameter values
# =============================================================================
# SALib sampled in log10 space, so we exponentiate to get actual values.

param_values_real = 10**param_values_log

print(f"\nFirst trajectory (17 runs):")
print(f"{'Run':<5}", end="")
for name in problem['names']:
    print(f"{name:>16}", end="")
print()
for i in range(17):
    print(f"{i:<5}", end="")
    for j in range(16):
        print(f"{param_values_real[i, j]:>16.4e}", end="")
    print()


First trajectory (17 runs):
Run      calcite_rate         fh_rmax      fh_k_donor           fh_SA         gt_rmax      gt_k_donor           gt_SA       root_resp        msr_rmax     msr_k_donor  msr_k_acceptor      denit_rmax   denit_k_donordenit_k_acceptor       aero_rate       aero_k_o2
0          7.1945e-07      2.5000e-06      4.6416e-06      5.0129e+06      2.5000e-08      4.6416e-07      2.8300e+07      7.9400e-14      2.5000e-08      5.0000e-05      2.1544e-04      1.0000e-05      1.0000e-04      4.6416e-06      4.6416e-10      2.1544e-04
1          7.1945e-07      2.5000e-06      4.6416e-06      5.0129e+06      2.5000e-08      4.6416e-07      1.3136e+06      7.9400e-14      2.5000e-08      5.0000e-05      2.1544e-04      1.0000e-05      1.0000e-04      4.6416e-06      4.6416e-10      2.1544e-04
2          7.1945e-07      2.5000e-06      4.6416e-06      5.0129e+06      2.5000e-08      4.6416e-07      1.3136e+06      7.9400e-14      2.5000e-08      5.0000e-05      2.1544e-04    

In [6]:
# =============================================================================
# Step 4: Save to CSV for downstream use
# =============================================================================
import pandas as pd

df = pd.DataFrame(param_values_real, columns=problem['names'])
df.index.name = 'run_id'
df.to_csv('/Users/christiandewey/Code/dewey-etal_meanders/sensitivity/morris_parameter_sets.csv')

print(f"\nSaved t_sets.csv")


Saved t_sets.csv


In [8]:
"""
Replace template placeholders in PFLOTRAN input files
with parameter values from the Morris sampling matrix.
"""
import shutil
from pathlib import Path

SENSITVITY_PATH="/Users/christiandewey/Code/dewey-etal_meanders/sensitivity/"

# Mapping from Morris sampling matrix column names to template placeholders.
# Keys must match the 'names' list in the SALib problem definition.
# Values must match the {{placeholder}} strings in the template files.
PARAM_TO_PLACEHOLDER = {
    'calcite_rate':   'calcite_rate',
    'fh_rmax':        'fh_rmax',
    'fh_k_donor':     'fh_k_donor',
    'fh_SA':          'fh_SA',
    'gt_rmax':        'gt_rmax',      # note: template uses gt_r_max
    'gt_k_donor':     'gt_k_donor',
    'gt_SA':          'gt_SA',
    'root_resp':      'root_resp',
    'msr_rmax':       'msr_rmax',
    'msr_k_donor':    'msr_k_donor',
    'msr_k_acceptor': 'msr_k_acceptor',
    'denit_rmax':     'denit_rmax',
    'denit_k_donor':  'denit_k_donor',
    'denit_k_acceptor': 'denit_k_acceptor',
    'aero_rate':      'aero_rate',
    'aero_k_o2':      'aero_k_o2',
}

# Template input filenames within each run directory
TEMPLATE_FILES = [
    'pflotran-mcp19_template_spin.in',
    'pflotran-mcp19_template.in',
]

def replace_placeholders(run_dir, param_names, param_values, run_index):
    """
    Replace {{placeholder}} strings in template input files with parameter values.

    Parameters
    ----------
    run_dir : Path
        Path to the run directory (e.g., /path/to/sensitivity/run001).
    param_names : list of str
        Parameter names matching the SALib problem definition 'names' list.
    param_values : array-like
        1D array of parameter values for this run (already in real space,
        not log10).
    run_index : int
        Run index (0-339), used to replace {{run_n}} in the main input file.
    """
    # Build replacement dictionary: placeholder string -> formatted value
    replacements = {}
    SA_PARAMS = {'fh_SA', 'gt_SA'}

    for name, value in zip(param_names, param_values):
        placeholder = PARAM_TO_PLACEHOLDER[name]
        formatted = f'{value:.6e}'
        if name not in SA_PARAMS:
            formatted = formatted.replace('e', 'd')
        replacements[f'{{{{{placeholder}}}}}'] = formatted
        # Also replace the run number placeholder in the main input file
    replacements['{{run_n}}'] = f'{run_index:03d}'

    for template_name in TEMPLATE_FILES:
        filepath = Path(run_dir, template_name)
        text = filepath.read_text()

        for placeholder, value in replacements.items():
            text = text.replace(placeholder, value)

        filepath.write_text(text)

"""
Copy template directory run_template/ to run001/ through run340/
for each Morris trajectory.
"""
# Number of Morris runs: 25 trajectories * (16 + 1) = 425
n_runs = 425

# Path to template directory
template_dir = Path(SENSITVITY_PATH, "run_template")

for i in range(0, n_runs):
    dest = Path(SENSITVITY_PATH, f"run{i:03d}")
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(template_dir, dest)
    replace_placeholders(dest, 
                         param_names=problem['names'], 
                         param_values=param_values_real[i,:],
                         run_index=i)
    for old_name, new_name in [
            ('pflotran-mcp19_template_spin.in', f'pflotran-mcp19_run{i:03d}_spin.in'),
            ('pflotran-mcp19_template.in',      f'pflotran-mcp19_run{i:03d}.in'),
        ]:
            Path(dest, old_name).rename(Path(dest, new_name))

print(f"Created run000 through run{n_runs - 1:03d}")

Created run000 through run424


In [9]:
"""
Create a PFLOTRAN run directory with default parameter values.
Use this to test spin-up duration before launching the full Morris analysis.
"""

from pathlib import Path
import shutil

SENSITIVITY_PATH = "/Users/christiandewey/Code/dewey-etal_meanders/sensitivity"

# Default parameter values (real space, not log10)
DEFAULTS = {
    'calcite_rate':     1.55e-06,
    'fh_rmax':          2.50e-06,
    'fh_k_donor':       1.00e-06,
    'fh_SA':            1.08e+07,
    'gt_rmax':          2.50e-07,
    'gt_k_donor':       1.00e-06,
    'gt_SA':            2.83e+06,
    'root_resp':        7.94328e-13,
    'msr_rmax':         2.50e-07,
    'msr_k_donor':      5.00e-06,
    'msr_k_acceptor':   1.00e-04,
    'denit_rmax':       1.00e-06,
    'denit_k_donor':    1.00e-05,
    'denit_k_acceptor': 1.00e-05,
    'aero_rate':        1.00e-09,
    'aero_k_o2':        1.00e-04,
}

# Surface area parameters use 'e' notation; all others use 'd' (Fortran)
SA_PARAMS = {'fh_SA', 'gt_SA'}

TEMPLATE_FILES = [
    'pflotran-mcp19_template_spin.in',
    'pflotran-mcp19_template.in',
]

# Copy template directory
template_dir = Path(SENSITIVITY_PATH, "run_template")
dest = Path(SENSITIVITY_PATH, "run_default")

if dest.exists():
    shutil.rmtree(dest)
shutil.copytree(template_dir, dest)

# Build replacements
replacements = {}
for name, value in DEFAULTS.items():
    formatted = f'{value:.6e}'
    if name not in SA_PARAMS:
        formatted = formatted.replace('e', 'd')
    replacements[f'{{{{{name}}}}}'] = formatted

replacements['{{run_n}}'] = 'default'

# Apply replacements
for template_name in TEMPLATE_FILES:
    filepath = Path(dest, template_name)
    text = filepath.read_text()

    for placeholder, value in replacements.items():
        text = text.replace(placeholder, value)

    filepath.write_text(text)

# Rename files
for old_name, new_name in [
    ('pflotran-mcp19_template_spin.in', 'pflotran-mcp19_default_spin.in'),
    ('pflotran-mcp19_template.in',      'pflotran-mcp19_default.in'),
]:
    Path(dest, old_name).rename(Path(dest, new_name))

print(f"Created {dest} with default parameter values")

Created /Users/christiandewey/Code/dewey-etal_meanders/sensitivity/run_default with default parameter values
